# Interactive experimentation

This notebook is designed to be used interactively. You can run the code snippets in the cells below and modify them as you like. The goal is to provide a hands-on experience with the concepts discussed in the course.

First, let's import the necessary libraries.

In [1]:
import os
import warnings

# Import necessary modules from LangChain and other libraries
from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_aws import BedrockLLM
from langchain_openai import OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_experimental.sql import SQLDatabaseChain
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain.agents.agent_types import AgentType
from langfuse.callback import CallbackHandler
from sqlalchemy import create_engine
from urllib.parse import quote_plus
# from presidio_analyzer import AnalyzerEngine
# from presidio_anonymizer import AnonymizerEngine

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

Then we set up the environment variables for the different services that we will be using.

In [2]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
KNOWLEDGE_BASE_CUSTOMER_SUPPORT_ID = os.environ["KNOWLEDGE_BASE_CUSTOMER_SUPPORT_ID"]
KNOWLEDGE_BASE_MARKETING_BOT_ID = os.environ["KNOWLEDGE_BASE_MARKETING_BOT_ID"]
LANGFUSE_PUBLIC_KEY = os.environ["LANGFUSE_PUBLIC_KEY"]
LANGFUSE_SECRET_KEY = os.environ["LANGFUSE_SECRET_KEY"]
AWS_REGION = "eu-central-1"
SCHEMA_NAME = "oreillyproductscrmdb"
S3_STAGING_DIR = "s3://oreillygenaiproductscrmdata/crm_data/"
LANGFUSE_HOST = "http://localhost:3000"

We can use `sqlalchemy` to connect to AWS Athena.

In [3]:
db_engine = create_engine("awsathena+rest://athena.{region_name}.amazonaws.com:443/{schema_name}?s3_staging_dir={s3_staging_dir}".format(
            region_name=AWS_REGION,
            schema_name=SCHEMA_NAME,
            s3_staging_dir=quote_plus(S3_STAGING_DIR)))

Langfuse setup, note: make sure it's running in the background.

In [4]:
# Initialize the callback handler using values from config
callback_handler = CallbackHandler(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST
)

In [5]:
# Create the database engine and SQLDatabase object
sql_database = SQLDatabase(db_engine)

In [6]:
# Create two retriever objects for different knowledge bases
customer_support_retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id=KNOWLEDGE_BASE_CUSTOMER_SUPPORT_ID,
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 4}},
)

marketing_bot_retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id=KNOWLEDGE_BASE_MARKETING_BOT_ID,
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 4}},
)

Initialize the two LLMs:

In [8]:
bedrock_llm = BedrockLLM(model_id="amazon.titan-text-express-v1")
openai_llm = OpenAI(
    temperature=0,
    verbose=True,
    openai_api_key=OPENAI_API_KEY
)

## Setup chains

Initialize SQL chain components:

In [9]:
sql_database_chain = SQLDatabaseChain(
    llm=openai_llm,
    database=sql_database,
    verbose=False
)
sql_toolkit = SQLDatabaseToolkit(db=sql_database, llm=openai_llm)
sql_agent_executor = create_sql_agent(
    llm=openai_llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    toolkit=sql_toolkit,
    max_iterations=15,
    max_execution_time=60,
    top_k=3,
    verbose=True
)

Set up the chat prompt template used for retrieval and create document Q&A and retrieval Q&A chains:

In [10]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
chat_prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", "{input}")]
)

document_qa_chain = create_stuff_documents_chain(bedrock_llm, chat_prompt_template)
retrieval_qa_chain = create_retrieval_chain(
    customer_support_retriever,
    document_qa_chain
)

Create chains for advice by classifying questions and then routing to specific chains:

In [11]:
# Classification chain to decide between products, customer reviews, or other topics.
classification_prompt = (
    "Given the user question below, classify it as either being about "
    "`products`, `customer reviews`, or `other`.\n\n"
    "Do not respond with more than one word.\n\n"
    "<question>\n{question}\n</question>\n\nClassification:"
)
classification_chain = (
    PromptTemplate.from_template(classification_prompt)
    | bedrock_llm
    | StrOutputParser()
)

# Product response chain.
product_prompt = (
    "You are an expert in products. Always answer questions starting with "
    '"Great that you ask about products!". Respond to the following question:\n\n'
    "Question: {question}\nAnswer:"
)
product_response_chain = PromptTemplate.from_template(product_prompt) | bedrock_llm

# Customer review response chain.
review_prompt = (
    "You are an expert in customer reviews. Always answer questions starting with "
    '"Great that you ask about customer reviews!". Respond to the following question:\n\n'
    "Question: {question}\nAnswer:"
)
review_response_chain = PromptTemplate.from_template(review_prompt) | bedrock_llm

# Fallback chain for queries that do not match.
fallback_prompt = "Respond that you cannot help with this query."
fallback_response_chain = PromptTemplate.from_template(fallback_prompt) | bedrock_llm

# Function to route advice based on topic.
def advice_router(payload):
    topic = payload.get("topic", "").lower()
    if "product" in topic:
        return product_response_chain
    elif "reviews" in topic:
        return review_response_chain
    else:
        return fallback_response_chain

# Compose the general advice chain.
general_advice_chain = (
    {"topic": classification_chain, "question": lambda payload: payload["question"]}
    | RunnableLambda(advice_router)
)

Build a chain to recommend an image based on a persona description and available images.

In [13]:
recommended_image_template = PromptTemplate.from_template(
    """You are helping select the best image to represent a persona.

    Persona description:
    {persona}

    Available images and captions:
    {caption_list}

    From the above, select the most fitting image filename and explain why.
    Return your answer in this format:

    Filename: <filename>
    Reason: <short explanation>
    """
)
recommended_image_chain = recommended_image_template | bedrock_llm

# Create chains for marketing tours.
marketing_tours_chain = create_stuff_documents_chain(bedrock_llm, chat_prompt_template)
recommended_tour_chain = create_retrieval_chain(
    marketing_bot_retriever,
    marketing_tours_chain
)

## Invoke chains

In [25]:
general_advice_chain.invoke({"question": "What are the features of a good product?"})

' Great that you ask about products! A good product has several key features that make it appealing to customers. These include:\n\n1. Quality: A good product is made with high-quality materials and components that are durable and reliable. It should be able to perform its intended function effectively and consistently over time.\n\n2. Functionality: A good product should be designed to meet the needs of its intended users. It should be easy to use, intuitive, and versatile, allowing users to accomplish their tasks efficiently and effectively.\n\n3. Design: A good product should have a visually appealing design that is eye-catching and memorable. It'

In [17]:
customer_support_retriever.invoke("what products do we have available?", config={"callbacks": [callback_handler]})

[Document(metadata={'location': {'s3Location': {'uri': 's3://oreillygenaiproducts/product_descriptions/utility_product_5.pdf'}, 'type': 'S3'}, 'score': 0.8836109042167589, 'type': 'TEXT'}, page_content="Product description     Here's a concise and realistic product description for the Miele Coffee Maker: **Introducing the Miele Coffee Maker** Elevate your morning routine with the Miele Coffee Maker, designed to deliver exceptional quality and convenience. This high-end coffee machine is crafted from premium materials, including stainless steel and glass, ensuring a durable and stylish appliance that withstands daily use. With its advanced brewing technology, the Miele Coffee Maker can produce a wide range of flavors and aromas, from smooth and balanced espresso shots to rich and velvety cappuccinos. The machine's intuitive interface makes it easy to customize your coffee experience, with adjustable settings for temperature, brewing time, and coffee strength. The Miele Coffee Maker is d

In [22]:
retrieval_qa_chain.invoke({"input": "what products do we have available?"})

{'input': 'what products do we have available?',
 'context': [Document(metadata={'location': {'s3Location': {'uri': 's3://oreillygenaiproducts/product_descriptions/utility_product_5.pdf'}, 'type': 'S3'}, 'score': 0.8836109042167589, 'type': 'TEXT'}, page_content="Product description     Here's a concise and realistic product description for the Miele Coffee Maker: **Introducing the Miele Coffee Maker** Elevate your morning routine with the Miele Coffee Maker, designed to deliver exceptional quality and convenience. This high-end coffee machine is crafted from premium materials, including stainless steel and glass, ensuring a durable and stylish appliance that withstands daily use. With its advanced brewing technology, the Miele Coffee Maker can produce a wide range of flavors and aromas, from smooth and balanced espresso shots to rich and velvety cappuccinos. The machine's intuitive interface makes it easy to customize your coffee experience, with adjustable settings for temperature, b

In [23]:
recommended_tour_chain.invoke({"input": "what is a good tour?"})

{'input': 'what is a good tour?',
 'context': [Document(metadata={'location': {'s3Location': {'uri': 's3://oreillygenaiproductsmarketingdata/tours/tours_dataset.json'}, 'type': 'S3'}, 'score': 0.6873371061277354, 'type': 'TEXT'}, page_content='This tour promotes sustainable practices and supports conservation efforts. A perfect getaway for nature lovers and adventure seekers!",       "price": 3599,       "duration_days": 10,       "start_date": "2024-09-15",       "features": [         "hiking",         "kayaking",         "wildlife"       ]     },     {       "id": 48,       "name": "Cultural Experience in Ecuador",       "region": "South America",       "description": "Immerse yourself in the diverse culture of Ecuador on this eco-adventure tour. Visit indigenous communities, explore historical sites, and experience traditional cuisine. This tour emphasizes sustainable practices and supports local economies while providing unforgettable experiences. Discover the rich traditions and s